In [1]:
import os
os.chdir('./../')
from datasets import load_dataset
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv()
load_dotenv(env_path)

/home/wisc/maheswararao/code/lightning-hydra-template/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [12]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

In [15]:
x = ['nirschl-lab/jung_et_al_2022',
'nirschl-lab/kather_et_al_2016',
'nirschl-lab/nirschl_et_al_2018',
'nirschl-lab/wong_et_al_2022',
'nirschl-lab/tang_et_al_2019',
'nirschl-lab/acevedo_et_al_2020']

In [ ]:
wong = load_dataset(x[3])
tang = load_dataset(x[4])

: 

In [13]:
# Custom Dataset wrapper
class HFDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image']  # PIL Image
        if self.transform:
            image = self.transform(image)
        return image

In [14]:
def build_transform(image_size: int = 299) -> transforms.Compose:
    """
    FID is computed with Inception-v3 features.
    Inception expects 299x299 and pixel values in [0,1].
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),                     # [0,1]
    ])

In [ ]:
wang_dataset = HFDataset(wong['train'], transform=build_transform(299))
tang_dataset = HFDataset(tang['train'], transform=build_transform(299))



In [9]:
#calculate FID between two datasets use gpu if available
from torchvision import transforms
import torch

to_tensor_transform = transforms.ToTensor()

fid = FrechetInceptionDistance(feature=2048).to('cuda' if torch.cuda.is_available() else 'cpu')
for img in wong['train']['image']:
    fid.update(to_tensor_transform(img).unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu'), real=True)
for img in tang['train']['image']:
    fid.update(to_tensor_transform(img).unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu'), real=False)
res = fid.compute()
print(res)

ValueError: Expecting image as torch.Tensor with dtype=torch.uint8

In [11]:
to_tensor_transform(img).shape

torch.Size([3, 256, 256])